In [1]:
from datasets import load_dataset

spanish_dataset = load_dataset("mteb/amazon_reviews_multi", "es")
english_dataset = load_dataset("mteb/amazon_reviews_multi", "en")
english_dataset

C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
})

In [2]:
def show_samples(datasets, num_samples=3, seed=42):
    samples = datasets['train'].shuffle(seed=seed).select(range(num_samples))
    for sample in samples:
        print(f"{sample['text']}")

In [3]:
show_samples(english_dataset)

Worked in front position, not rear

3 stars because these are not rear brakes as stated in the item description. At least the mount adapter only worked on the front fork of the bike that I got it for.
meh

Does it’s job and it’s gorgeous but mine is falling apart, I had to basically put it together again with hot glue
Can't beat these for the money

Bought this for handling miscellaneous aircraft parts and hanger "stuff" that I needed to organize; it really fit the bill. The unit arrived quickly, was well packaged and arrived intact (always a good sign). There are five wall mounts-- three on the top and two on the bottom. I wanted to mount it on the wall, so all I had to do was to remove the top two layers of plastic drawers, as well as the bottom corner drawers, place it when I wanted and mark it; I then used some of the new plastic screw in wall anchors (the 50 pound variety) and it easily mounted to the wall. Some have remarked that they wanted dividers for the drawers, and that the

In [4]:
english_dataset.set_format('pandas')
english_df = english_dataset['train'][:]
english_dataset.reset_format()

english_df

,id,text,label,label_text
0,en_0964290,I'll spend twice the amount of time boxing up ...,0,0
1,en_0690095,Not use able\n\nthe cabinet dot were all detac...,0,0
2,en_0311558,The product is junk.\n\nI received my first or...,0,0
3,en_0044972,Fucking waste of money\n\nThis product is a pi...,0,0
4,en_0784379,bubble\n\nwent through 3 in one day doesn't fi...,0,0
...,...,...,...,...
199995,en_0046316,"Nice and fit as advertised\n\nCute slippers, m...",4,4
199996,en_0956024,good to keep the kids engaged\n\nMy 6 year old...,4,4
199997,en_0589358,This works\n\nReplaced my battery with it. Wor...,4,4
199998,en_0970602,"Well made.\n\nI like them, holding up well.",4,4


In [5]:
def split_title_body(example):
    title, _, body = example['text'].partition("\n\n")
    return {"review_title": title, "review_body": body, "stars": example["label"] + 1}

In [6]:
spanish_dataset = spanish_dataset.map(split_title_body, remove_columns=["text", "label", "label_text"])
english_dataset = english_dataset.map(split_title_body, remove_columns=["text", "label", "label_text"])
english_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 5000
    })
})

In [7]:
from datasets import concatenate_datasets, DatasetDict

def sample(ds, n_train=10000, n_eval=500, seed=42):
    return DatasetDict({
        "train": ds["train"].shuffle(seed=seed).select(range(n_train)),
        "validation": ds["validation"].shuffle(seed=seed).select(range(n_eval)),
        "test": ds["test"].shuffle(seed=seed).select(range(n_eval)),
    })

english_reviews = sample(english_dataset)
spanish_reviews = sample(spanish_dataset)

In [8]:
reviews_dataset = DatasetDict()

for split in english_reviews.keys():
    reviews_dataset[split] = concatenate_datasets([english_reviews[split], spanish_reviews[split]])
    reviews_dataset[split] = reviews_dataset[split].shuffle(seed=42)

In [9]:
reviews_dataset = reviews_dataset.filter(
    lambda x: len(x["review_title"].split()) > 2 and len(x["review_body"].strip()) > 0
)
reviews_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 11447
    })
    validation: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 565
    })
    test: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 549
    })
})

In [10]:
from transformers import AutoTokenizer

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_input_length = 512
max_target_length = 30

In [11]:
def preprocess_function(examples):
    model_inputs = tokenizer(examples["review_body"], max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=examples["review_title"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [12]:
tokenized_datasets = reviews_dataset.map(
    preprocess_function, batched=True, remove_columns=reviews_dataset["train"].column_names
)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 11447
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 565
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 549
    })
})

In [13]:
%pip install rouge_score

Note: you may need to restart the kernel to use updated packages.


In [14]:
import evaluate

metric = evaluate.load('rouge')

In [15]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

In [16]:
scores = metric.compute(
    predictions=[generated_summary], references=[reference_summary]
)
scores

{'rouge1': np.float64(0.923076923076923),
 'rouge2': np.float64(0.7272727272727272),
 'rougeL': np.float64(0.923076923076923),
 'rougeLsum': np.float64(0.923076923076923)}

In [17]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [18]:
import nltk

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [19]:
from nltk.tokenize import sent_tokenize

def three_summary_sentence(text):
    return "\n".join(sent_tokenize(text)[:3])

print(three_summary_sentence(reviews_dataset["train"][1]["review_body"]))

Producto que sirve de adaptador para poder conectar nuestra Wii via HDMI.
Lo de los 720p de resolución es.
Un timo ya que la Wii no llega a esta resolución y el adaptador no hace escalado.La resolución máxima que podremos obtener configurando desde rl menú de Wii es de quinientos y pico,pero se ve bastante bien.


In [20]:
def evaluate_baseline(dataset, metric):
    summaries = [three_summary_sentence(text) for text in dataset["review_body"]]
    return metric.compute(predictions=summaries, references=dataset["review_title"])

In [21]:
import pandas as pd

score = evaluate_baseline(reviews_dataset["validation"], metric)
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, round(score[rn] * 100, 2)) for rn in rouge_names)
rouge_dict

{'rouge1': np.float64(19.22),
 'rouge2': np.float64(10.92),
 'rougeL': np.float64(17.72),
 'rougeLsum': np.float64(18.23)}

In [23]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("mt5-small-finetuned-amazon-en-es/checkpoint-11448")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 240.24it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [23]:
from huggingface_hub import notebook_login

notebook_login()

In [27]:
from transformers import Seq2SeqTrainingArguments

batch_size = 8
num_train_epochs = 8
# Show the training loss with every epoch
logging_steps = len(tokenized_datasets["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

args = Seq2SeqTrainingArguments(
    output_dir=f"{model_name}-finetuned-amazon-en-es",
    eval_strategy="epoch",
    learning_rate=5.6e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    logging_steps=logging_steps,
    push_to_hub=False,
)

In [28]:
import numpy as np

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = ['\n'.join(sent_tokenize(sentence.strip())) for sentence in decoded_preds]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]

    result = metric.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )

    result = {key: value * 100 for key, value in result.items()}
    return {k: round(v, 4) for k, v in result.items()}

In [29]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [30]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,8.004440,3.068172,15.678200,6.351600,15.310100,15.319500
2,3.760823,2.928660,16.153900,6.597600,15.694700,15.678900
3,3.422697,2.814656,18.246900,9.552200,17.766500,17.695500
4,3.260544,2.774701,17.777400,9.297900,17.342100,17.267200
5,3.153625,2.778807,18.325400,9.742400,17.737500,17.719000
6,3.100019,2.755807,19.429600,10.523800,18.750100,18.685000
7,3.049556,2.749435,19.258900,10.429900,18.656800,18.600600


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]
C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\transformers\generation\utils.py:1955: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.60s/it]


In [31]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
No log,2.746948,0,19.568000,10.741100,18.939200,18.949500


{'eval_loss': 2.746948480606079,
 'eval_rouge1': 19.568,
 'eval_rouge2': 10.7411,
 'eval_rougeL': 18.9392,
 'eval_rougeLsum': 18.9495}